In [0]:
%run "../notebooks/helper_functions"

In [0]:
# Create widgets for configuration
dbutils.widgets.text("catalog_name", "novacart_catalog", "1. Target Catalog Name")
dbutils.widgets.text("bronze_schema", "bronze_schema", "2. Bronze Schema Name")
dbutils.widgets.text("silver_schema", "silver_schema", "3. Silver Schema Name")
dbutils.widgets.text("job_run_id", None, "4. Job Run ID from Databricks Workflow Job")

# Read widget values
catalog_name = dbutils.widgets.get("catalog_name").strip()
bronze_schema = dbutils.widgets.get("bronze_schema").strip()
silver_schema = dbutils.widgets.get("silver_schema").strip()
job_run_id = dbutils.widgets.get("job_run_id").strip()

# Construct table names
bronze_prefix = f"{catalog_name}.{bronze_schema}"
silver_prefix = f"{catalog_name}.{silver_schema}"
silver_control_table = f"{catalog_name}.{silver_schema}.processing_control"

print("=" * 60)
print("Silver Configuration:")
print("=" * 60)
print(f"Target Catalog:        {catalog_name}")
print(f"Bronze Schema:         {bronze_schema}")
print(f"Silver Schema:         {silver_schema}")
print("=" * 60)
print(f"Bronze Prefix:         {bronze_prefix}")
print(f"Silver Prefix:         {silver_prefix}")
print(f"Silver Control Table:  {silver_control_table}")
print("=" * 60)

#### Imports and setup

This cell imports helper functions and creates a silver_run_id for the current run

In [0]:
from pyspark.sql.window import Window
import uuid

In [0]:
if job_run_id is not None:
  silver_run_id = job_run_id
else:
  # Not running as a job, generate a UUID
  silver_run_id = str(uuid.uuid4())
  print(f"Running interactively, generated UUID: {silver_run_id}")

print(f"Current silver run id: {silver_run_id}")

#### Orders incremental processing
This cell processes **orders** from Bronze to Silver.
It does the following
- reads only new Bronze order rows
- cleans values like order_status and order_amount
- keeps only the latest version per order_id
- validates business rules
- sends bad rows to quarantine
- merges good rows into orders_transformed

In [0]:
df_raw = spark.sql(f"select * from {bronze_prefix}.orders_raw").limit(50)
display(df_raw)

In [0]:
# Read only the Bronze order rows that Silver has not processed
orders_inc, last_orders_ingested_at, last_orders_run_id = get_incremental_bronze(
    f"{bronze_prefix}.orders_raw",
    "orders",
    silver_control_table
)

# Count the incremental order rows entering Silver in this run
orders_inc_count = orders_inc.count()
print(f"orders rows_to_process_in_silver: {orders_inc_count}")

# Only run Silver order cleaning and validation when there are new Bronze order rows
if orders_inc_count > 0:
  # Create a window that keeps the latest order for each order_id
  order_window = Window.partitionBy("order_id").orderBy(
    F.col("updated_at").cast("timestamp").desc(),
    F.col("bronze_ingested_at").desc()
  )
  
  # Start the silver order-cleaning pipeline. This block standardizes and deduplicates raw order records.
  orders_cleaned = (
    orders_inc
    # Standardize order_status to uppercase so values such as shipped and SHIPPED become consistent
    .withColumn("order_status", F.upper(F.trim(F.col("order_status"))))
    .withColumn("order_status", F.when(F.col("order_status") == "", F.lit(None)).otherwise(F.col("order_status")))
    # Remove formatting characters from order_amount so it can be cast to a numeric type
    .withColumn("order_amount", F.regexp_replace(F.col("order_amount"), r"[$, ]", ""))
    .withColumn("order_amount", F.when(F.trim(F.col("order_amount")).isin("N/A", "NULL", "??", ""), None).otherwise(F.col("order_amount")))
    .withColumn("order_amount", F.col("order_amount").cast("double"))
    .withColumn("created_at", F.to_timestamp("created_at"))
    .withColumn("updated_at", F.to_timestamp("updated_at"))
    # Assign a row number inside each business key so we can keep only the latest version of that record
    .withColumn("row_rank", F.row_number().over(order_window))
    # Keep only the latest record for each business key.
    .filter(F.col("row_rank") == 1)
    .drop("row_rank")
    .withColumn("silver_run_id", F.lit(silver_run_id))
  )

  # Merge the cleaned or validated Silver dataset into its Delta target table
  upsert_to_silver(
    orders_cleaned,
    f"{silver_prefix}.orders_cleaned",
    ["order_id"]
  )

  # Apply silver data-quality rules to the cleaned order records
  orders_validated = (
    orders_cleaned
    .withColumn(
      "to_be_verified_by_orders_team",
      F.when(F.col("customer_id").isNull(), "verify_customer_id")
      .when(F.col("product_id").isNull(), "verify_product_id")
      .when(F.col("order_status").isNull() | (F.trim(F.col("order_status")) == ""), "verify_order_status")
      .when(F.col("order_amount").isNull() | (F.col("order_amount") <= 0), "verify_order_amount")
      .otherwise("No issues")     
    )
  .withColumn(
    "check_order_amount",
    F.when(F.col("order_amount").isNull() | (F.col("order_amount") <= 0), F.lit(True))
    .otherwise(F.lit(False))
  )
  .withColumn("order_date", F.to_date("created_at"))
  .withColumn("order_year", F.year("created_at"))
  .withColumn("order_month", F.month("created_at"))
  .withColumn("order_day", F.dayofmonth("created_at"))
  .withColumn("order_dow", F.date_format("created_at", "E"))
  )

  # Keep only valid order rows for the transformed silver table
  orders_good_df = orders_validated.filter(F.col("to_be_verified_by_orders_team") == "No issues")
  # Send invalid order rows to the quarantine dataset for manual review
  orders_bad = (
    orders_validated
    .filter(F.col("to_be_verified_by_orders_team") != "No issues")
    .withColumn("quarantine_ts", F.current_timestamp())
  )


  # Merge the cleaned or validated silver dataset into its Delta target table.
  upsert_to_silver(
    orders_good_df,
    f"{silver_prefix}.orders_transformed",
    ["order_id"]
  )

  # Append bad order rows to the quarantine table instead of losing them
  orders_bad.write.format("delta").mode("append").saveAsTable(f"{silver_prefix}.orders_quarantine")

  mx_ingested = orders_inc.agg(F.max("bronze_ingested_at").alias("mx")).collect()[0]["mx"]

  mx_run_id = (
    orders_inc.filter(F.col("bronze_ingested_at") == F.lit(mx_ingested))
    .agg(F.max("bronze_run_id").alias("mx"))
    .collect()[0]["mx"]
  )

  orders_good_df_count = orders_good_df.count()

  upsert_silver_control(
      "orders",
      mx_run_id,
      mx_ingested,
      orders_good_df_count,
      silver_run_id,
      silver_control_table
  )

else:
  print("No new orders Bronze rows for Silver")
  
  upsert_silver_control(
    "orders",
    last_orders_run_id,
    last_orders_ingested_at,
    orders_inc_count,
    silver_run_id,
    silver_control_table
  )

#### Products incremental processing
This cell processes **products** from Bronze to Silver
It handles 
- product name cleanup
- category standardization
- price cleanup and numeric conversion
- latest-record selection per product_id
- data quality validation
- quarantine for bad rows
- merge into Silver current-state tables

In [0]:
# Step 5 - Products incremental processing
# Read only the Bronze product rows that Silver have not processed yet
products_inc, last_products_ingested_at, last_products_run_id = get_incremental_bronze(
    f"{bronze_prefix}.products_raw",
    "products",
    silver_control_table
)

# Count the incremental product rows entering Silver in this run.
products_inc_count = products_inc.count()
print(f"products row_to_process_in_silver = {products_inc_count}")

if products_inc_count > 0:
  # Create a window that keeps the latest product record for each product_id
  product_window = Window.partitionBy("product_id").orderBy(
    F.col("updated_at").cast("timestamp").desc(),
    F.col("bronze_ingested_at").desc()
  )

  # Start the silver product-cleaning pipeline. This block standardizes and deduplicates raw product records.
  products_cleaned = (
    products_inc
    # Standardize product_name by trimming spaces and converting text to uppercase.
    .withColumn("product_name", F.upper(F.trim(F.col("product_name"))))
    .withColumn("product_name", F.regexp_replace(F.col("product_name"), r"[-_]", " "))
    .withColumn("product_name", F.when(F.col("product_name") == "", F.lit(None)).otherwise(F.col("product_name")))
    .withColumn(
      "category",
      F.when(F.upper(F.trim(F.col("category"))).contains("ELECTRONICS"), "ELECTRONICS")
      .otherwise(F.upper(F.trim(F.col("category"))))
    )
    # Start cleaning the product price field before converting it to numeric
    .withColumn("price", F.trim(F.col("price")))
    .withColumn("price", F.regexp_replace(F.col("price"), r"\$", ""))
    .withColumn("price", F.regexp_replace(F.col("price"), ",", "."))
    .withColumn("price", F.regexp_replace(F.col("price"), r"\s+", ""))
    .withColumn("price", F.expr("try_cast(price as double)"))
    .withColumn("updated_at", F.to_timestamp("updated_at"))
    # Assign a row number inside each business key so we can keep only the latest version of that record
    .withColumn("row_rank", F.row_number().over(product_window))
    # Keep only the latest record for each business key
    .filter(F.col("row_rank") == 1)
    .drop("row_rank")
    .withColumn("silver_run_id", F.lit(silver_run_id))
    )


  # Merge the cleaned or validated silver dataset into its Delta target table
  upsert_to_silver(
      products_cleaned,
      f"{silver_prefix}.products_cleaned",
      ["product_id"]
  )

  # Apply silver data-quality rules to the cleaned product records
  products_validated = (
    products_cleaned
    .withColumn(
      "to_be_verified_by_products_team",
      F.when(F.col("product_name").isNull(), "verify_product_name")
      .when(F.col("category").isNull(), "verify_category")
      .when(F.col("price").isNull() | (F.col("price") <= 0), "verify_price")
      .otherwise("No issues")
    )
    .withColumn(
      "check_product_price",
      F.when(F.col("price").isNull() | (F.col("price") <= 0), "invalid_price")
      .otherwise("valid_price")
    )
  )

  # Keep only valid product rows for the transformed Silver table
  products_good = products_validated.filter(
    (F.col("to_be_verified_by_products_team") == "No issues") &
    (F.col("check_product_price") == "valid_price")
  )
  if "price_raw" in products_good.columns:
    # Keep only valid product rows for the transformed Silver table
    products_good = products_good.drop("price_raw")

  # Send invalid product rows to the quarantined dataset for manual review
  products_bad = products_validated.filter(
    (F.col("to_be_verified_by_products_team") != "No issues") |
    (F.col("check_product_price") == "invalid_price")
  )

  # Merge the cleaned or validated silver dataset into its delta target table
  upsert_to_silver(
      products_good,
      f"{silver_prefix}.products_transformed",
      ["product_id"]
  )
  # Append bad products rows to the quarantine table instead of losing them
  products_bad.write.format("delta").mode("append").saveAsTable(f"{silver_prefix}.products_quarantine")

  mx_ingested = products_inc.agg(F.max("bronze_ingested_at").alias("mx")).collect()[0]["mx"]
  mx_run = products_inc.filter(F.col("bronze_ingested_at") == F.lit(mx_ingested)).agg(F.max("bronze_run_id").alias("mx")).collect()[0]["mx"]
  
  upsert_silver_control(
      "products",
      mx_run,
      mx_ingested,
      products_good.count(),
      silver_run_id,
      silver_control_table
  )
else:
  print("No new product bronze rows for Silver.")
  upsert_silver_control(
    "products",
    last_products_run_id,
    last_products_ingested_at,
    products_inc_count,
    silver_run_id,
    silver_control_table
  )

#### Payments incremental processing
This cell processes **payments** from Bronze to Silver.

It cleans
- payment_status
- paid_amount
- processed_at

Then it validates records, quarantines bad rows, and merges valid rows into the Silver transformed payments table.


In [0]:
# Step 6 - Payments incremental processing
# Read only the bronze payment rows that Silver has not processed yet.
payments_inc, last_payments_ingested_at, last_payments_run_id = get_incremental_bronze(
    f"{bronze_prefix}.payments_raw",
    "payments",
    silver_control_table
)
print("Payments last processed Bronze ingested at =", last_payments_ingested_at)
# Count the incremental payment rows entering silver in this run
payments_inc_count = payments_inc.count()
print(f"payments rows_to_process_in_silver = {payments_inc_count}")

if payments_inc_count > 0:
    # Create a window that keeps the latest payment record for each payment_id
    payment_window = Window.partitionBy("payment_id").orderBy(
        F.col("processed_at").cast("timestamp").desc(),
        F.col("bronze_ingested_at").desc()
    )

    # Start the silver payment-cleaning pipeline. This block standardizes and deduplicates raw payment records
    payments_cleaned = (
        payments_inc
        .withColumn("payment_status", F.upper(F.trim(F.col("payment_status"))))
        .withColumn("payment_status", F.when(F.col("payment_status") == "", F.lit(None)).otherwise(F.col("payment_status")))
        # Start cleaning the payment amount field before converting it to numeric.
        .withColumn("paid_amount", F.trim(F.col("paid_amount")))
        .withColumn("paid_amount", F.regexp_replace(F.col("paid_amount"), r"\$" , ""))
        .withColumn("paid_amount", F.regexp_replace(F.col("paid_amount"), ",", "."))
        .withColumn("paid_amount", F.regexp_replace(F.col("paid_amount"), r"\s+", ""))
        .withColumn("paid_amount", F.expr("try_cast(paid_amount as double)"))
        .withColumn("processed_at", F.to_timestamp("processed_at"))
        # Assign a row number inside each business key so we can keep only the latest version of that record
        .withColumn("row_rank", F.row_number().over(payment_window))
        # Keep only the latest record for each business key.
        .filter(F.col("row_rank") == 1)
        .drop("row_rank")
        .withColumn("silver_run_id", F.lit(silver_run_id))
    )

    # Merge the cleaned or validated silver dataset into its delta target table
    upsert_to_silver(
        payments_cleaned,
        f"{silver_prefix}.payments_cleaned",
        ["payment_id"]
    )

    # Apply Silver data-quality rules to the cleaned payment records
    payments_validated = (
        payments_cleaned
        .withColumn(
            "to_be_verified_by_payments_team",
            F.when(F.col("order_id").isNull(), "verify_order_id")
            .when(F.col("payment_status").isNull(), "verify_payment_status")
            .when(F.col("paid_amount").isNull() | (F.col("paid_amount") <= 0), "verify_paid_amount")
            .otherwise("No Issues")
        )
        .withColumn("check_paid_amount",
            F.when(F.col("paid_amount").isNull() | (F.col("paid_amount") <= 0), F.lit(True)).otherwise(F.lit(False))
        )
    )

    # Keep only valid payment rows for the transformed Silver table
    payments_good = payments_validated.filter(F.col("to_be_verified_by_payments_team") == "No Issues")
    # Send invalid payment rows to the quarantine dataset for manual review
    payments_bad = payments_validated.filter(F.col("to_be_verified_by_payments_team") != "No Issues") \
                .withColumn("quarantine_ts", F.current_timestamp())

    # Merge the cleaned or validated Silver dataset into its Delta target table
    upsert_to_silver(
        payments_good,
        f"{silver_prefix}.payments_transformed",
        ["payment_id"]
    )
    # Append bad payment rows to the quarantine table instead of losing them
    payments_bad.write.format("delta").mode("append").saveAsTable(f"{silver_prefix}.payments_quarantine")

    mx_ingested = payments_inc.agg(F.max("bronze_ingested_at").alias("mx")).collect()[0]["mx"]
    mx_run = payments_inc.filter(F.col("bronze_ingested_at") == F.lit(mx_ingested)).agg(F.max("bronze_run_id").alias("mx")).collect()[0]["mx"]
    
    upsert_silver_control(
        "payments",
        mx_run,
        mx_ingested,
        payments_good.count(),
        silver_run_id,
        silver_control_table
    )

else:
    print("No new payments Bronze rows for Silver.")
    upsert_silver_control(
        "payments",
        last_payments_run_id,
        last_payments_ingested_at,
        payments_inc_count,
        silver_run_id,
        silver_control_table
    )

#### Quick validation
This final cell prints Silver transformed row counts and shows the Silver control table, so you can confirm the incremental processing behavior

In [0]:
print("Products transformed count: ", spark.sql(f"SELECT COUNT(*) FROM {silver_prefix}.products_transformed").collect()[0][0])

print("Orders transformed count: ", spark.sql(f"SELECT COUNT(*) FROM {silver_prefix}.orders_transformed").collect()[0][0])

print("Payments transformed count: ", spark.sql(f"SELECT COUNT(*) FROM {silver_prefix}.payments_transformed").collect()[0][0])

display(spark.table(silver_control_table).orderBy("entity_name"))